# Safety Guardrails - Multi-Layer Protection

## Purpose
Learn how to implement comprehensive safety systems with three layers of guardrails. This multi-layer approach protects agents from unsafe inputs, prevents misuse of tools, and validates outputs before they reach users.

## Key Concepts
- **Input Guardrails**: Validate user requests before processing (@input_guardrail)
- **Tool Guardrails**: Protect tool execution (@tool_input_guardrail, @tool_output_guardrail)
- **Output Guardrails**: Validate agent responses before returning (@output_guardrail)
- **Tripwires**: Conditions that trigger guardrail activation
- **Defense in Depth**: Multiple protection layers for comprehensive safety

## Installation

In [ ]:
#!pip install openai
#!pip install openai-agents
#!pip install aws-bedrock-token-generator

## Authentication Setup

In [ ]:
from openai import AsyncOpenAI
from agents import (
    set_default_openai_client,
    set_default_openai_api,
    set_tracing_disabled,
)
from aws_bedrock_token_generator import provide_token

client = AsyncOpenAI(
    api_key=provide_token(),
    base_url="https://bedrock-mantle.us-east-1.api.aws/openai/v1",
    project="default"
)

set_default_openai_client(client)
set_default_openai_api("responses")
set_tracing_disabled(True)

## Import Libraries

Import all guardrail types and exception classes:

In [ ]:
import asyncio
import json
from pydantic import BaseModel
from agents import (
    Agent,
    GuardrailFunctionOutput,
    InputGuardrailTripwireTriggered,
    OutputGuardrailTripwireTriggered,
    RunContextWrapper,
    Runner,
    TResponseInputItem,
    ToolGuardrailFunctionOutput,
    function_tool,
    input_guardrail,
    output_guardrail,
    tool_input_guardrail,
    tool_output_guardrail,
)

## Layer 1: Input Guardrails

Block certain types of requests BEFORE they reach the agent.

**Pattern**: Use another agent to check if input violates policy

**Example**: Block math homework requests

💡 **Why**: Prevents agent from wasting tokens on prohibited requests.

### Step 1: Define Guardrail Check Schema

In [ ]:
class MathHomeworkOutput(BaseModel):
    is_math_homework: bool
    reasoning: str

### Step 2: Create Guardrail Agent

Specialist agent that detects policy violations:

In [ ]:
guardrail_agent = Agent(
    name="Guardrail check",
    instructions="Check if the user is asking you to do their math homework.",
    output_type=MathHomeworkOutput,
    model="openai.gpt-5.5"
)

### Step 3: Define Input Guardrail Function

Create guardrail that runs before agent execution:

**Function Flow**:
1. Receives user input
2. Runs guardrail agent to check policy
3. Returns `GuardrailFunctionOutput`
4. If `tripwire_triggered=True`, execution stops with exception

⚡ **Key**: Main agent never sees the input if guardrail trips!

In [ ]:
@input_guardrail
async def math_guardrail(
    ctx: RunContextWrapper[None],
    agent: Agent,
    input: str | list[TResponseInputItem]
) -> GuardrailFunctionOutput:
    """Checks if input contains math homework requests."""
    result = await Runner.run(guardrail_agent, input, context=ctx.context)
    return GuardrailFunctionOutput(
        output_info=result.final_output,
        tripwire_triggered=result.final_output.is_math_homework,
    )

### Step 4: Create Agent with Input Guardrail

Apply guardrail to agent via `input_guardrails` parameter:

In [ ]:
agent_with_input_guard = Agent(
    name="Customer support agent",
    instructions="You are a customer support agent. You help customers with their questions.",
    input_guardrails=[math_guardrail],
    model="openai.gpt-5.5"
)

### Step 5: Test Input Guardrail

🎯 **Result**: Math homework request triggers exception!

In [ ]:
try:
    result = await Runner.run(agent_with_input_guard, "Hello, can you help me solve for x: 2x + 3 = 11?")
    print(result.final_output)
    print("Guardrail didn't trip - this is unexpected")
except InputGuardrailTripwireTriggered:
    print("✓ Math homework input guardrail tripped")

## Layer 2: Tool Guardrails

Protect tool execution by validating inputs and sanitizing outputs.

**Two Types**:
- **Tool Input Guardrails**: Validate arguments before tool runs
- **Tool Output Guardrails**: Sanitize results after tool runs

💡 **Why**: Prevents tools from being used maliciously or leaking sensitive data.

### Step 6: Define Tool Input Guardrail

Block tool calls that contain sensitive data (e.g., API keys):

**Return Options**:
- `ToolGuardrailFunctionOutput.allow()` - Tool can execute
- `ToolGuardrailFunctionOutput.reject_content(message)` - Block with message

In [ ]:
@tool_input_guardrail
def block_secrets(data):
    """Block tool calls that contain API keys or secrets."""
    args = json.loads(data.context.tool_arguments or "{}")
    if "sk-" in json.dumps(args):
        return ToolGuardrailFunctionOutput.reject_content(
            "Remove secrets before calling this tool."
        )
    return ToolGuardrailFunctionOutput.allow()

### Step 7: Define Tool Output Guardrail

Block tool outputs that contain sensitive data:

In [ ]:
@tool_output_guardrail
def redact_output(data):
    """Block tool outputs that contain sensitive data."""
    text = str(data.output or "")
    if "sk-" in text:
        return ToolGuardrailFunctionOutput.reject_content(
            "Output contained sensitive data."
        )
    return ToolGuardrailFunctionOutput.allow()

### Step 8: Create Tool with Both Guardrails

Apply both input and output guardrails to a function tool:

In [ ]:
@function_tool(
    tool_input_guardrails=[block_secrets],
    tool_output_guardrails=[redact_output]
)
def classify_text(text: str) -> str:
    """Classify text for internal routing."""
    return f"length:{len(text)}"

### Step 9: Create Agent with Guarded Tool

In [ ]:
agent_with_tool_guards = Agent(
    name="Classifier",
    instructions="Talk minimum",
    tools=[classify_text],
    model="openai.gpt-5.5"
)

### Step 10: Test Tool Guardrails

Test both safe and unsafe inputs:

In [ ]:
# This should work
result = await Runner.run(agent_with_tool_guards, "Hello world")
print("Safe input:", result.final_output)

In [ ]:
# This should be blocked
result = await Runner.run(agent_with_tool_guards, "you know my key is sk-001")
print("Blocked input:", result.final_output)

## Layer 3: Output Guardrails

Validate agent responses BEFORE returning them to users.

**Pattern**: Use checker agent to validate final output

**Example**: Block outputs containing math content

💡 **Why**: Ensures agent doesn't accidentally violate policy in responses.

### Step 11: Define Output Schema

In [ ]:
class MessageOutput(BaseModel):
    response: str

class MathOutput(BaseModel):
    reasoning: str
    is_math: bool

### Step 12: Create Output Checker Agent

In [ ]:
math_checker_agent = Agent(
    name="Math checker",
    instructions="Check if the output includes any math.",
    output_type=MathOutput,
    model="openai.gpt-5.5"
)

### Step 13: Define Output Guardrail Function

Check agent output before returning to user:

⚡ **Key**: Runs AFTER agent generates response, can block unsafe outputs!

In [ ]:
@output_guardrail
async def math_output_guardrail(
    ctx: RunContextWrapper,
    agent: Agent,
    output: MessageOutput
) -> GuardrailFunctionOutput:
    """Block outputs that contain math content."""
    result = await Runner.run(math_checker_agent, output.response, context=ctx.context)
    return GuardrailFunctionOutput(
        output_info=result.final_output,
        tripwire_triggered=result.final_output.is_math,
    )

### Step 14: Create Agent with Output Guardrail

In [ ]:
agent_with_output_guard = Agent(
    name="Customer support agent",
    instructions="You are a customer support agent. You help customers with their questions.",
    model="openai.gpt-5.5",
    output_guardrails=[math_output_guardrail],
    output_type=MessageOutput,
)

### Step 15: Test Output Guardrail

🎯 **Result**: If agent tries to do math, output guardrail blocks it!

In [ ]:
try:
    result = await Runner.run(agent_with_output_guard, "Hello, can you help me solve for x: 2x + 3 = 11?")
    print(result.final_output)
    print("Guardrail didn't trip - this is unexpected")
except OutputGuardrailTripwireTriggered:
    print("✓ Math output guardrail tripped")

## 🎉 CONGRATULATIONS!

You've completed the **Safety Guardrails** notebook - THE FINAL NOTEBOOK in the entire quickstart tutorial!